# 02 — Cleaning Pipeline

**Owner:** Tech Lead / ETL Owner
**Input:** Raw dataframe from `01_extraction.ipynb`
**Output:** `data/processed/online_retail_II_clean.csv`
**Rubric link:** Data Quality & ETL (15 marks)

## Purpose

Apply and **document** every transformation. Every cleaning decision must be justified in a markdown cell directly above the code — "we did X because the profile showed Y." This is the single most-audited notebook in the repo.

## Cleaning steps (planned)

1. Standardize column names to snake_case
2. Parse types (`InvoiceDate` → datetime, `Customer ID` → nullable int)
3. Drop rows missing critical fields (InvoiceDate, StockCode)
4. Flag returns (`is_return`) — don't delete them
5. Flag registered customers (`is_registered`)
6. Flag non-product stock codes (`is_product`)
7. Compute `line_revenue` = Quantity × Price
8. Clean `description` (strip, standardize nulls)
9. Standardize country names (`country_clean`)
10. Add time features (year-month, hour, day-of-week)
11. Add coarse product category hint (`product_category_hint`)
12. Drop exact duplicates

Each step below gets its own cell + a written justification.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Re-load from 01 (cheapest path) — or import from scripts/etl_pipeline.py
RAW_PATH = Path("../data/raw/online_retail_II.xlsx")
xl = pd.ExcelFile(RAW_PATH)
df = pd.concat([xl.parse(sheet_name=s) for s in xl.sheet_names], ignore_index=True)
n_start = len(df)
print(f"Starting rows: {n_start:,}")

### Step 1 — Standardize column names

Original columns have spaces and mixed case. Converting to snake_case makes downstream code cleaner and matches the production `etl_pipeline.py`.

In [ ]:
df.columns = (df.columns.str.strip()
                        .str.replace(" ", "_")
                        .str.replace("-", "_")
                        .str.lower())
df.columns.tolist()

### Step 2 — Parse types

- `InvoiceDate` → pandas datetime (needed for time features)
- `Quantity` / `Price` → numeric (coerce errors to NaN so we can find bad rows)
- `Customer ID` → nullable integer (pandas `Int64`) — avoids the `123.0` float display

In [ ]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")
df["quantity"]    = pd.to_numeric(df["quantity"], errors="coerce")
df["price"]       = pd.to_numeric(df["price"], errors="coerce")
df["customer_id"] = df["customer_id"].astype("Int64")
df.dtypes

### Step 3 — Drop rows missing critical fields

A row without InvoiceDate or StockCode cannot be used for any analysis. These should be rare; if it's >0.1% of rows, investigate before dropping.

In [ ]:
crit = df["invoicedate"].isna() | df["stockcode"].isna()
print(f"Rows missing critical fields: {int(crit.sum()):,} ({crit.mean()*100:.3f}%)")
df = df[~crit].copy()
print(f"Rows remaining: {len(df):,}")

### Step 4 — Flag returns

Returns are *legitimate business events* — we net them against gross revenue rather than deleting them. Two signals:
1. `Invoice` starts with 'C' (explicit cancellation)
2. `Quantity < 0` (return line)

In [ ]:
df["is_return"] = (
    df["invoice"].astype(str).str.startswith("C")
    | (df["quantity"] < 0)
)
print(f"Return rows: {int(df['is_return'].sum()):,} ({df['is_return'].mean()*100:.2f}%)")

### Step 5 — Flag registered customers

~25% of rows have no `Customer ID` (guest checkouts). We cannot do RFM/cohort analysis on these, but they are real revenue — flag rather than drop.

In [ ]:
df["is_registered"] = df["customer_id"].notna()
print(f"Registered rows: {df['is_registered'].mean()*100:.2f}%")

### Step 6 — Flag non-product stock codes

Codes like `POST`, `M`, `D`, `BANK CHARGES` are adjustments/services, not products. Keep in the dataset for gross revenue but exclude from product-level analysis.

In [ ]:
NON_PRODUCT = {"POST","D","M","BANK CHARGES","AMAZONFEE","DOT","CRUK","PADS","S"}
df["is_product"] = ~df["stockcode"].astype(str).str.upper().isin(NON_PRODUCT)
df.groupby("is_product").size()

### Step 7 — Line revenue

In [ ]:
df["line_revenue"] = df["quantity"] * df["price"]
df["line_revenue"].describe()

### Step 8 — Clean description

Strip whitespace; convert empty strings / 'nan' / 'None' to actual NaN.

In [ ]:
df["description"] = df["description"].astype(str).str.strip()
df.loc[df["description"].isin(["","nan","None"]), "description"] = np.nan
print("Descriptions missing:", df["description"].isna().sum())

### Step 9 — Standardize country names

In [ ]:
COUNTRY_MAP = {
    "EIRE": "Ireland",
    "RSA": "South Africa",
    "USA": "United States",
    "Channel Islands": "United Kingdom",  # crown dependency
}
df["country_clean"] = df["country"].replace(COUNTRY_MAP)
df["country_clean"].value_counts().head(10)

### Step 10 — Time features

In [ ]:
df["invoice_year_month"] = df["invoicedate"].dt.to_period("M").astype(str)
df["invoice_hour"]        = df["invoicedate"].dt.hour
df["invoice_day_of_week"] = df["invoicedate"].dt.day_name()
df[["invoicedate","invoice_year_month","invoice_hour","invoice_day_of_week"]].head()

### Step 11 — Coarse category hint

Keyword-based mapping. **Not authoritative** — disclose this as a limitation in the report.

In [ ]:
def category_hint(desc):
    if not isinstance(desc, str): return "unknown"
    d = desc.lower()
    for cat, kws in {
        "christmas": ["christmas","xmas","advent"],
        "bag":       ["bag","jumbo","lunch box"],
        "mug_cup":   ["mug","cup","teacup"],
        "candle":    ["candle","t-light","lantern"],
        "decor":     ["decoration","ornament","hanging"],
        "paper":     ["paper","napkin","card"],
        "kitchen":   ["cake","bake","kitchen","apron"],
        "garden":    ["garden","plant","flower"],
        "kids":      ["toy","kids","children","doll"],
    }.items():
        if any(kw in d for kw in kws): return cat
    return "other"

df["product_category_hint"] = df["description"].apply(category_hint)
df["product_category_hint"].value_counts()

### Step 12 — Drop exact duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["invoice","stockcode","quantity","price","invoicedate"])
print(f"Dropped {before - len(df):,} exact duplicates")

## Final summary

In [ ]:
print(f"Final rows: {len(df):,} (from {n_start:,}, {len(df)/n_start*100:.1f}% retained)")
print(f"Columns: {list(df.columns)}")

In [ ]:
OUT = Path("../data/processed/online_retail_II_clean.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print(f"Wrote {OUT}")